# Question 1 – Planifier la réalisation d'un produit (3 points)

> **Mode d'emploi du book.** 
> - Exécutez les cellules dans l'ordre (`Maj + Entrée`).
> - Les cellules marquées `# A COMPLETER` sont à compléter ; les cases **Réponse** sont à rédiger.

---

## Le problème

Dix tâches, nommées de **A à J**, sont nécessaires à la réalisation d'un produit.

| Tâche | A | B | C | D | E | F | G | H | I | J |
|-------|---|---|---|---|---|---|---|---|---|---|
| Durée (jours) | 7 | 3 | 1 | 8 | 2 | 1 | 1 | 3 | 2 | 5 |

Les tâches sont liées par des **précédences** : une tâche ne peut commencer qu'une fois ses prédécesseurs terminés.

- B et D sont exécutées après A ;
- C est exécutée après B ;
- E, F et G sont exécutées après C **et** D ;
- H est exécutée après F ;
- I est exécutée après H ;
- J est exécutée après E, G **et** I.

**Objectif :** trouver une date de début pour chaque tâche, de sorte que le produit soit terminé **le plus tôt possible**.

> **Commencez par un dessin.** 
> - Représentez les tâches et les précédences par un graphe (une flèche de X vers Y si Y est exécutée après X). 
> - Ce dessin vous servira à écrire les contraintes **et** à vérifier le résultat du solveur.

**Q1.0 – Analyse.**   
Compléter ce diagramme mermaid suivant (le diagramme s'affiche seul, sinon, installer le plugin mermaid dans votre IDE).
Double cliquez pour éditer.

```mermaid
flowchart LR
  A["A (7 j)"] --> B["B (3 j)"] --> C["C (1 j)"]
  A --> D["D (8 j)"]
```

## L'exemple fourni

Le fichier `exemple_planification.py` (portage de l'exemple `planif_taches_td` de  `ProductionScheduling-ort.py` vu en cours) planifie 6 tâches
numérotées.   
Voici sa version avec des **tableaux de données** et une **minimisation** de la date de fin :

In [3]:
import inspect
import exemple_planification
print(inspect.getsource(exemple_planification.planif_optimale))

def planif_optimale():
    """Version avec tableaux et minimisation de la date de fin."""
    model = cp_model.CpModel()
    nb_taches = 6
    temps_max = 12
    duree = [1, 1, 2, 1, 2, 1]
    # (avant, apres) : 'apres' ne commence qu'une fois 'avant' terminee
    precedences = [(0, 5), (1, 3), (2, 3), (3, 4), (4, 5)]

    debut = [model.new_int_var(0, temps_max, f"debut_{t}") for t in range(nb_taches)]
    for avant, apres in precedences:
        model.add(debut[apres] >= debut[avant] + duree[avant])

    fin = model.new_int_var(0, temps_max, "fin")
    model.add(fin == debut[5] + duree[5])      # la tache 5 est la derniere
    model.minimize(fin)

    solver = cp_model.CpSolver()
    statut = solver.solve(model)
    if statut == cp_model.OPTIMAL:
        print("Fin au plus tot :", solver.value(fin))
        for t in range(nb_taches):
            print(f"  tache {t} : debut {solver.value(debut[t])}, fin {solver.value(debut[t]) + duree[t]}")
    else:
        print("pas de solution :",

In [7]:
#execution 
exemple_planification.planif_optimale()

Fin au plus tot : 6
  tache 0 : debut 0, fin 1
  tache 1 : debut 0, fin 1
  tache 2 : debut 0, fin 2
  tache 3 : debut 2, fin 3
  tache 4 : debut 3, fin 5
  tache 5 : debut 5, fin 6


## Modéliser

**Q1.1 – Les données.** 
- Représentez les noms, les durées et les précédences par des structures Python.
- But: avoir un code générique et lisible. 

In [ ]:
from ortools.sat.python import cp_model
#tableau des noms des tâches
NOMS = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]

#dictionnaire "Tache": durée
DUREE = {"A": 7, "B": 3}   # A COMPLETER

#tableau de tupes de précédences ("TacheAvant": "TacheApres")
PRECEDENCES = [("A", "B"), ("A", "D")]  # A COMPLETER 

assert len(DUREE) == len(NOMS), "il manque des durees"
print(len(PRECEDENCES), "precedences")

**Q1.2 – Les variables.** 
- Placez les variables de début de tâches dans un dictionnaire `debut{nom:variable contrainte entiere}`.  
Ainsi `debut["A"]` retourne la variable entiere correspondant au debut de la tâche.
  - Quel domaine choisir pour les tâches ? La borne supérieure doit être sûre (ne jamais exclure une solution) sans être inutilement grande.

**Q1.3 – Les contraintes** Les précédences : « Y est exécutée après X » se traduit par : début de Y ≥ début de X + durée de X.
Une boucle sur `PRECEDENCES` suffit, comme dans l'exemple.

**Q1.4 – L'objectif.** Ici une seule tâche finale, "J"; décrivez l'objectif en ce sens.

In [ ]:
model = cp_model.CpModel()

HORIZON = ...                                # A COMPLETER
debut = {t: model.new_int_var(0, HORIZON, f"debut_{t}") for t in NOMS}

# precedences
# A COMPLETER

# fin du projet = la plus grande des dates de fin, a minimiser
fin_projet = model.new_int_var(0, HORIZON, "fin_projet")
# A COMPLETER

solver = cp_model.CpSolver()
statut = solver.solve(model)
print(solver.status_name(statut))
if statut == cp_model.OPTIMAL:
    print("Fin du projet au plus tot : jour", solver.value(fin_projet))

<details>
<summary><b>Indice : l'horizon</b></summary>

Dans le pire des cas, on exécute toutes les tâches les unes après les autres.

</details>

## Présenter le résultat

Affichez, pour chaque tâche, ses dates de début et de fin, puis un **diagramme de Gantt** en texte. Par exemple,
pour l'exemple fourni :

```text
  0 |=
  1 |=
  2 |==
  3 |  =
  4 |   ==
  5 |     =
```

In [ ]:
def gantt(debuts):
    # debuts : dictionnaire nom de tache -> date de debut
    # A COMPLETER
    ...

gantt({t: solver.value(debut[t]) for t in NOMS})

<details>
<summary><b>Indice : une ligne par tâche</b></summary>

`print(f"  {t} |{' ' * debuts[t]}{'=' * DUREE[t]}")` : des espaces jusqu'au début, puis un caractère par jour.

</details>

**(R) Q1.5 –** Quelle est la date de fin minimale du projet ?   
- Quelles tâches ne peuvent pas être retardées sans retarder tout le projet (le **chemin critique**) ?   
- Vérifiez votre réponse sur votre dessin.

** Réponse à Q2.6 :**

*(double-cliquez ici pour écrire)*

---

## Intervalles 

**Q 1.6 Ressources limitées.**  
- Chaque tâche mobilise un technicien et l'équipe n'en compte que deux. 
- Dans votre algorithme de recherche de solution, 
  - Créez un intervalle par tâche avec `model.new_fixed_size_interval_var(debut[t], DUREE[t], f"tache_{t}")` 
  - puis posez la contrainte `model.add_cumulative(intervalles, [1] * len(NOMS), 2)`. 
- La date de fin change-t-elle ? Et avec un seul technicien ?
